In [ ]:
import sys
import os
import time

import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.fft import fft
from scipy.signal import find_peaks
from scipy.stats import skew

sys.path.insert(0, '../../..') # Load Parent directory to import utils
from utils.constants import Configs

: 

### Load Data From cache

In [ ]:
cache_path = Configs.ICATER_WRIST_ONLY_FILTERED_DATA_DIR

if not os.path.exists(cache_path):
    raise FileNotFoundError(f"Processed data not found at {cache_path}. Please run the data processing notebook first.")
df = pd.read_csv(cache_path)

In [ ]:
print(df.head())

In [ ]:
def calculate_sparc(speed_profile, fs=60.0):
    """
    Calculates the SPARC (Spectral Arc Length) - The best metric to evaluate the "sinuosity" of speed
    - speed_profile: 1D array of the 'vel' column
    - fs: Sampling frequency in Hz (assumes 50Hz, change if your data is different)
    """
    if len(speed_profile) < 2: return 0
    # Normalization
    speed_profile = speed_profile / speed_profile.max()
    N = len(speed_profile)
    # FFT resolution 
    Nfft = int(2**(np.ceil(np.log2(N)) + 4))
    
    speed_spectrum = np.abs(fft(speed_profile, Nfft))
    freq = np.linspace(0, fs/2, Nfft//2)
    speed_spectrum = speed_spectrum[:Nfft//2] / speed_spectrum.max()
    
    # Corte a 10Hz (frequência limite para movimentos humanos normais)
    fc_idx = np.searchsorted(freq, 10.0)
    spectrum_cut = speed_spectrum[:fc_idx]
    
    # Calculate the arc length
    df = (freq[1] - freq[0])
    ds = np.sqrt(df**2 + np.diff(spectrum_cut)**2)
    return -np.sum(ds)

def calculate_ldlj(jerk_profile, duration_sec, distance):
    """
    Calculates LDLJ (Log Dimensionless Jerk) - metric for smoothness of movement
    - jerk_profile: 1D array of the 'jerk' column
    - duration_sec: duration of the movement in seconds
    - distance: distance traveled in meters
    """
    if distance <= 0 or duration_sec <= 0:
        return 0
    
    dt = duration_sec / len(jerk_profile)
    integral_jerk_sq = np.sum(jerk_profile**2) * dt
    
    dsj = (duration_sec**5 / distance**2) * integral_jerk_sq
    
    return -np.log(dsj) if dsj > 0 else 0

def count_zero_crossings(acc_profile, noise_threshold=0.01):
    """
    Remove small noises around 0 to count real changes in acceleration to deceleration.
    """
    # Only consider that the acceleration changed sign if it passes a minimum threshold (e.g., 0.01 m/s^2)
    acc_clean = np.where(np.abs(acc_profile) > noise_threshold, acc_profile, 0)
    # Search where the signal passes from positive to negative (and vice-versa), ignoring the zeros in between
    acc_clean = acc_clean[acc_clean != 0] 
    if len(acc_clean) < 2:
        return 0
    return len(np.where(np.diff(np.sign(acc_clean)))[0])

### Do Feature Extraction for all movements and save them to cache

In [ ]:
all_features = {}
times = []
fs = Configs.FS
dt = 1.0 / fs

for movement in ["Pick", "Move", "Place"]:
    print(f"Extracting features for movement: {movement}")
    start_time = time.perf_counter()

    cache_path = Configs.wrist_only_feature_extraction_features(movement)
    mov_data = df[df['movLabel'] == movement]
    groups = mov_data.groupby(['sub', 'repID', 'movID'])
    features_list = []

    for (sub, repID, movID), group in groups:
        # Garantir ordenação temporal
        group = group.sort_values(by='step')

        # Spatial dimensions & Extrema
        duration_frames = len(group)
        duration_sec = duration_frames * dt

        range_x = group['x'].max() - group['x'].min()
        range_y = group['y'].max() - group['y'].min()
        range_z = group['z'].max() - group['z'].min()

        # Spatial efficiency
        start_point = group[['x', 'y', 'z']].iloc[0].values
        end_point = group[['x', 'y', 'z']].iloc[-1].values
        straight_line_dist = np.linalg.norm(end_point - start_point)
        path_length = np.sum(np.sqrt(np.sum(np.diff(group[['x', 'y', 'z']].values, axis=0)**2, axis=1)))
        
        path_ratio = (path_length / straight_line_dist) if straight_line_dist > 0 else 1.0

        # Velocity and Acceleration profiles
        vel_array = group['vel'].values
        acc_array = group['acc'].values
        jerk_array = group['jerk'].values
        
        max_vel = vel_array.max()
        mean_vel = vel_array.mean()
        max_acc = acc_array.max()
        mean_acc = acc_array.mean()
        max_jerk = np.abs(jerk_array).max()

        # Profile shape & Planning
        time_to_peak_vel = np.argmax(vel_array) * dt
        mean_to_peak_vel_ratio = (mean_vel / max_vel) if max_vel > 0 else 0
        vel_skewness = skew(vel_array)
        
        peaks, _ = find_peaks(vel_array, prominence=max_vel*0.10)
        n_velocity_peaks = len(peaks)

        # Smoothness & Complexity
        sparc_smoothness = calculate_sparc(vel_array, fs)
        ldlj = calculate_ldlj(jerk_array, duration_sec, path_length)
        
        total_curvature = group['k'].sum() 
        total_torsion = group['t'].sum()

        features_list.append({
            'sub': sub, 'repID': repID, 'movID': movID, 'movLabel': movement,
            'duration': duration_sec,
            'range_x': range_x, 'range_y': range_y, 'range_z': range_z,
            'path_length': path_length,
            'path_ratio': path_ratio,
            'mean_vel': mean_vel,
            'max_vel': max_vel,
            'time_to_peak_vel': time_to_peak_vel,
            'mean_to_peak_vel_ratio': mean_to_peak_vel_ratio,
            'vel_skewness': vel_skewness,
            'n_velocity_peaks': n_velocity_peaks,
            'mean_acc': mean_acc,
            'max_acc': max_acc,
            'max_jerk': max_jerk,
            'ldlj': ldlj,
            'sparc': sparc_smoothness,
            'total_curvature': total_curvature,
            'total_torsion': total_torsion
        })
        
    features_df = pd.DataFrame(features_list)
    all_features[movement] = features_df
    
    end_time = time.perf_counter()
    elapsed = end_time - start_time
    times.append(elapsed)

    # Export all features to cache
    print(f"Features extracted in {elapsed:.2f} seconds.")
    joblib.dump(features_df, cache_path)

In [ ]:
times

In [ ]:
all_features["Pick"][["sub", "repID", "movID", "movLabel", "path_length"]]

## Plot Correlation Matrix

In [ ]:
all_features_array = pd.concat(all_features.values(), ignore_index=True)
# MOVEMEN = "Pick" 

fig, ax = plt.subplots(figsize=(8, 8))
sns.heatmap(all_features_array[Configs.FEATURE_EXTRACTION_COLS_TO_TRAIN].corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
plt.show()

Foi utilizado o filtro de correlação de Pearson para fazer a seleção das features

In [ ]:
X = all_features_array.drop(columns=['sub', 'repID', 'movID', 'movLabel']) # 
corr_matrix = X.corr().abs()

# Find exact pairs of highly correlated features
highly_correlated_pairs = []
threshold = 0.85

for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if corr_matrix.iloc[i, j] > threshold:
            feature_1 = corr_matrix.columns[i]
            feature_2 = corr_matrix.columns[j]
            correlation_val = corr_matrix.iloc[i, j]
            highly_correlated_pairs.append((feature_1, feature_2, correlation_val))

In [ ]:
META_COLS = ['sub', 'repID', 'movID', 'movLabel']
FEATURE_COLS = [c for c in all_features_array.columns if c not in Configs.META_COLS] # To manually put in constants.py
print(len(FEATURE_COLS), FEATURE_COLS)